# Pythia-70M projected optimizer-state probe

Coupling-Phase Spectroscopy — governed Colab runner.

Runs the minimal real-model CPS probe. It uses a deterministic text batch and reconstructed Adam moments. This validates the full model → loss → optimizer map → JVP → projected operator → phase sweep pipeline, but it is not an exact historical optimizer-state claim.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[pythia,notebooks]"], check=True)
print("repo", repo, "ref", GIT_REF)

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available(): print("device", torch.cuda.get_device_name(0))

In [ ]:
from cps.pythia.config import load_probe_config
from cps.pythia.runner import run_probe
config = load_probe_config("subjects/pythia/configs/pythia_70m_smoke.yaml")
output = run_probe(config)
print(output)

In [ ]:
import json, pathlib
root = pathlib.Path(output)
manifest = json.loads((root / "manifest.json").read_text())
records = json.loads((root / "couplings.json").read_text())
print(json.dumps({"projection": manifest["projection"], "top_couplings": records[:3]}, indent=2))

In [ ]:
import pathlib, shutil
export_dir = pathlib.Path("/content/cps-export")
export_dir.mkdir(parents=True, exist_ok=True)
source = pathlib.Path("/content/cps-artifacts")
if source.exists():
    shutil.copytree(source, export_dir / "artifacts", dirs_exist_ok=True)
shutil.make_archive("/content/cps-export", "zip", "/content/cps-export")
print("exported", export_dir)